In [1]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import sys
import numpy as np
import pickle

sys.path.append('D:\\Program\\MyCode\\Round_robin_SL\\Round-Robin')
from models import *
from clients_datasets import *
from tqdm.notebook import tqdm
from utils import *
from AttFunc import *
from Fisher_LeNet import *

In [2]:
from PoisonedFL.torchVesion import poisonedfl

In [3]:
# 初始化攻击相关参数
history = None  # 初始化历史梯度
fixed_rand = torch.randn(1204682)  # 根据模型参数维度初始化
last_grad = None  # 初始化上一轮梯度
scaling_factor = 100000.0

In [8]:
# 初始化攻击相关参数
history = None  # 初始化历史梯度
fixed_rand = torch.randn(1204682)  # 根据模型参数维度初始化
last_grad = None  # 初始化上一轮梯度
scaling_factor = 100000.0

#   攻击变量在循环内动态初始化，随机向量和历史每轮可能变化，攻击更随机不稳定
iters = 10
for iter in tqdm(range(iters), desc="Training", unit="iter"):
    # 初始化训练
    batch_size = 600
    epochs = 30
    NC = 10
    dataset = 'mnist'
    clients_trainloader = load_clients_trainsets(dataset, NC, batch_size)
    clients_testloader = load_clients_testsets(dataset, NC, batch_size)
    server, server_opt, clients, clients_opts = set_model_and_opt(dataset, NC)
    client_level = 1
    server_level = 4
    criterion = torch.nn.CrossEntropyLoss()

    # 训练
    mal_client_id = iter
    server.train()
    for i in range(NC):
        clients[i].train()
    server.apply(init_weights)
    clients[0].apply(init_weights)
    last_trained_params = clients[0].state_dict()
    for epoch in range(epochs):
        for idx, client in enumerate(clients):
            client.load_state_dict(last_trained_params)
            for j, data in enumerate(clients_trainloader[idx]):
                # 正常训练部分
                images, labels = data
                images = images.cuda()
                labels = labels.cuda()
                smashed_data = client.forward(images, client_level=client_level)
                output = server.forward(smashed_data, server_level=server_level)
                clients_opts[idx].zero_grad()
                server_opt.zero_grad()
                loss = criterion(output, labels)
                loss.backward()
                clients_opts[idx].step()
                server_opt.step()
            # 权重共享
            last_trained_params = client.state_dict()
            # 攻击部分
            if idx == mal_client_id:
                # 获取当前模型参数
                current_model = [param.data.clone() for param in client.parameters()]
                if history is None:
                    history = torch.cat([param.view(-1) for param in current_model])
                if last_grad is None:
                    last_grad = torch.cat([torch.zeros_like(param.view(-1)) for param in current_model])

                # 动态调整 fixed_rand 的大小
                total_params = sum(param.numel() for param in client.parameters())  # 计算模型参数总大小
                if fixed_rand.numel() != total_params:  # 检查 fixed_rand 的大小是否匹配
                    fixed_rand = torch.randn(total_params)  # 重新初始化 fixed_rand

                # 调用攻击函数
                mal_update, scaling_factor = poisonedfl(
                    v=[torch.zeros_like(param.view(-1)) for param in current_model],
                    net=client,
                    lr=0.01,
                    nfake=1,
                    history=history,
                    fixed_rand=fixed_rand,
                    init_model=current_model,
                    last_50_model=current_model,
                    last_grad=last_grad,
                    e=epoch,
                    scaling_factor=scaling_factor
                )

                if isinstance(mal_update, list):
                    device = mal_update[0].device
                    mal_update = torch.cat([t.to(device).reshape(-1) for t in mal_update if t is not None])
                    mal_update = mal_update.to(client.parameters().__next__().device)

                # 更新恶意客户端参数
                idx = 0
                for param in client.parameters():
                    param.data += mal_update[idx:idx + param.numel()].view(param.size())
                    idx += param.numel()
                # 更新历史梯度
                last_grad = torch.cat([
                    param.grad.view(-1) if param.grad is not None else torch.zeros_like(param.data).view(-1)
                    for param in client.parameters()
                ])

Training:   0%|          | 0/10 [00:00<?, ?iter/s]

In [10]:
    # 第一个 测试部分
    # 测试部分
    ID_acc1 = []
    clients_acc1 = []
    server.eval()
    for i in range(NC):
        clients[i].eval()
    with torch.no_grad():
        for idx, client in enumerate(clients):
            correct = 0
            total = 0
            acc1 = 0
            for data in clients_testloader[idx]:
                images, labels = data
                images, labels = images.cuda(), labels.cuda()

                smashed_data = client.forward(images, client_level=client_level)
                output = server.forward(smashed_data, server_level=server_level)
                _, pre = torch.max(output.data, 1)
                total += images.shape[0]
                correct += (pre == labels).sum().item()
            acc1 = 100 * correct / total
            print(f"Client {idx} Accuracy after attack: {acc1:.2f}%")
            clients_acc1.append(acc1)
    acc1 = np.mean(clients_acc1)
    ID_acc1.append(acc1)
    print(f"Iteration {iter} - Average Accuracy: {acc1:.2f}%")

print("Final Results:")
print(ID_acc1)

Client 0 Accuracy after attack: 97.40%
Client 1 Accuracy after attack: 97.30%
Client 2 Accuracy after attack: 96.60%
Client 3 Accuracy after attack: 97.10%
Client 4 Accuracy after attack: 97.10%
Client 5 Accuracy after attack: 97.30%
Client 6 Accuracy after attack: 98.00%
Client 7 Accuracy after attack: 96.60%
Client 8 Accuracy after attack: 97.10%
Client 9 Accuracy after attack: 96.20%
Iteration 9 - Average Accuracy: 97.07%
Final Results:
[97.07000000000001]


In [5]:
# 攻击变量在循环外统一初始化，固定随机向量和梯度历史，攻击更稳定持续。
iters = 10
for iter in tqdm(range(iters), desc="Training", unit="iter"):
    # 初始化训练
    batch_size = 600
    epochs = 30
    NC = 10
    dataset = 'mnist'
    clients_trainloader = load_clients_trainsets(dataset, NC, batch_size)
    clients_testloader = load_clients_testsets(dataset, NC, batch_size)
    server, server_opt, clients, clients_opts = set_model_and_opt(dataset, NC)
    client_level = 1
    server_level = 4
    criterion = torch.nn.CrossEntropyLoss()

    # 训练
    mal_client_id = iter
    server.train()
    for i in range(NC):
        clients[i].train()
    server.apply(init_weights)
    clients[0].apply(init_weights)
    last_trained_params = clients[0].state_dict()

    # 初始化攻击变量
    history = None
    last_grad = None
    fixed_rand = None
    scaling_factor = 100000.

    for epoch in range(epochs):
        for idx, client in enumerate(clients):
            client.load_state_dict(last_trained_params)
            for j, data in enumerate(clients_trainloader[idx]):
                images, labels = data
                images = images.cuda()
                labels = labels.cuda()
                smashed_data = client.forward(images, client_level=client_level)
                output = server.forward(smashed_data, server_level=server_level)
                clients_opts[idx].zero_grad()
                server_opt.zero_grad()
                loss = criterion(output, labels)
                loss.backward()
                clients_opts[idx].step()
                server_opt.step()

            # 权重共享
            last_trained_params = client.state_dict()

            # 攻击部分
            if idx == mal_client_id:
                # 获取当前模型参数
                current_model = [param.data.clone() for param in client.parameters()]
                total_params = sum(param.numel() for param in client.parameters())
                device = next(client.parameters()).device

                # 构造 last_grad: [total_params]
                last_grad = torch.cat([
                    param.grad.view(-1) if param.grad is not None else torch.zeros_like(param.data).view(-1)
                    for param in client.parameters()
                ])

                # 构造 history: [total_params, 1]
                history = last_grad.clone().view(-1, 1)

                # 初始化或修正 fixed_rand
                if fixed_rand is None or fixed_rand.numel() != total_params:
                    fixed_rand = torch.randn(total_params, device=device)
                else:
                    fixed_rand = fixed_rand.to(device)

                # 调用攻击函数
                mal_update, scaling_factor = poisonedfl(
                    v=[torch.zeros_like(param.view(-1)) for param in current_model],
                    net=client,
                    lr=0.01,
                    nfake=1,
                    history=history,
                    fixed_rand=fixed_rand,
                    init_model=current_model,
                    last_50_model=current_model,
                    last_grad=last_grad,
                    e=epoch,
                    scaling_factor=scaling_factor
                )

                # 拼接 mal_update
                if isinstance(mal_update, list):
                    mal_update = torch.cat([t.reshape(-1) for t in mal_update if t is not None])
                    mal_update = mal_update.to(device)

                # 更新恶意客户端模型参数
                idx_ = 0
                for param in client.parameters():
                    numel = param.numel()
                    param.data += mal_update[idx_:idx_ + numel].view(param.size())
                    idx_ += numel


Training:   0%|          | 0/10 [00:00<?, ?iter/s]

In [7]:
    # 测试部分
    ID_acc1 = []
    clients_acc1 = []
    server.eval()
    for i in range(NC):
        clients[i].eval()
    with torch.no_grad():
        for idx, client in enumerate(clients):
            correct = 0
            total = 0
            acc1 = 0
            for data in clients_testloader[idx]:
                images, labels = data
                images, labels = images.cuda(), labels.cuda()

                smashed_data = client.forward(images, client_level=client_level)
                output = server.forward(smashed_data, server_level=server_level)
                _, pre = torch.max(output.data, 1)
                total += images.shape[0]
                correct += (pre == labels).sum().item()
            acc1 = 100 * correct / total
            print(f"Client {idx} Accuracy after attack: {acc1:.2f}%")
            clients_acc1.append(acc1)
    acc1 = np.mean(clients_acc1)
    ID_acc1.append(acc1)
    print(f"Iteration {iter} - Average Accuracy: {acc1:.2f}%")

print("Final Results:")
print(ID_acc1)

Client 0 Accuracy after attack: 91.40%
Client 1 Accuracy after attack: 91.40%
Client 2 Accuracy after attack: 90.40%
Client 3 Accuracy after attack: 90.60%
Client 4 Accuracy after attack: 92.30%
Client 5 Accuracy after attack: 92.20%
Client 6 Accuracy after attack: 91.60%
Client 7 Accuracy after attack: 91.80%
Client 8 Accuracy after attack: 91.80%
Client 9 Accuracy after attack: 89.80%
Iteration 9 - Average Accuracy: 91.33%
Final Results:
[91.33]
